In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
ratings = pd.read_csv("Ratings.csv", sep=",", quotechar='"', encoding='latin-1', low_memory=False)
ratings = ratings[ratings['Book-Rating'] > 0]  # filter out zero ratings

users = pd.read_csv("Users.csv", sep=",", quotechar='"', encoding='latin-1', low_memory=False)

books = pd.read_csv("Books.csv", sep=",", quotechar='"', encoding='latin-1', low_memory=False)

In [8]:
print(ratings.head())

   User-ID        ISBN  Book-Rating
1   276726  0155061224            5
3   276729  052165615X            3
4   276729  0521795028            6
6   276736  3257224281            8
7   276737  0600570967            6


In [10]:
print(users.head())

   User-ID                            Location   Age
0        1                  nyc, new york, usa   NaN
1        2           stockton, california, usa  18.0
2        3     moscow, yukon territory, russia   NaN
3        4           porto, v.n.gaia, portugal  17.0
4        5  farnborough, hants, united kingdom   NaN


In [12]:
print(books.head())

         ISBN                                         Book-Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

            Book-Author Year-Of-Publication                   Publisher  \
0    Mark P. O. Morford                2002     Oxford University Press   
1  Richard Bruce Wright                2001       HarperFlamingo Canada   
2          Carlo D'Este                1991             HarperPerennial   
3      Gina Bari Kolata                1999        Farrar Straus Giroux   
4       E. J. W. Barber                1999  W. W. Norton &amp; Company   

                                         Image-URL-S  \
0  http://images.amazon.com/images/P/0195153448.0...   
1  http://images.amazon.com/

In [14]:
active_users = ratings['User-ID'].value_counts()[ratings['User-ID'].value_counts() >= 10].index
popular_books = ratings['ISBN'].value_counts()[ratings['ISBN'].value_counts() >= 10].index

filtered_ratings = ratings[
    ratings['User-ID'].isin(active_users) & ratings['ISBN'].isin(popular_books)
]

In [16]:
print("Null Values in ratings",ratings.isnull().sum())

Null Values in ratings User-ID        0
ISBN           0
Book-Rating    0
dtype: int64


In [18]:
train_data, test_data=train_test_split(filtered_ratings,test_size=0.25, random_state=42)
train_data=train_data.reset_index(drop=True)
test_data=test_data.reset_index(drop=True)

In [20]:
print("Train set size", train_data.shape)
print("Test set size", test_data.shape)

Train set size (67917, 3)
Test set size (22639, 3)


In [22]:
print(train_data.head())

   User-ID        ISBN  Book-Rating
0   250709  0425165701            5
1   141089  0671534645            8
2   129851  0345337662            7
3   211426  0060976241            9
4   254196  0449218473            3


In [26]:
test_data = test_data[
    test_data['User-ID'].isin(train_data['User-ID']) &
    test_data['ISBN'].isin(train_data['ISBN'])
].reset_index(drop=True)

In [28]:
user_item_matrix=train_data.pivot_table(index='User-ID', columns='ISBN', values='Book-Rating')
user_item_matrix_filled=user_item_matrix.fillna(0)

In [30]:
item_user_matrix=user_item_matrix_filled.T
item_similarity=cosine_similarity(item_user_matrix)

In [32]:
item_similarity_df = pd.DataFrame(item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index)

In [38]:
from tqdm import tqdm
import numpy as np

In [40]:
def predict_rating(user_id, isbn, user_item_matrix, similarity_df, k=5):
    if isbn not in similarity_df.index or user_id not in user_item_matrix.index:
        return np.nan  # cannot predict

    # Books the user has rated
    user_ratings = user_item_matrix.loc[user_id]
    rated_books = user_ratings[user_ratings > 0]

    # Similarities to other books
    similar_books = similarity_df[isbn].drop(isbn, errors='ignore')
    common_books = similar_books[rated_books.index.intersection(similar_books.index)]

    if common_books.empty:
        return np.nan

    # Top-k similar books
    top_k = common_books.sort_values(ascending=False).head(k)
    top_k_ratings = rated_books[top_k.index]

    # Weighted average
    numerator = np.dot(top_k.values, top_k_ratings.values)
    denominator = np.sum(np.abs(top_k.values))
    if denominator == 0:
        return np.nan
    return numerator / denominator

def predict_test_ratings(test_df, user_item_matrix, similarity_df, k=5):
    predictions = []
    actuals = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        user = row['User-ID']
        isbn = row['ISBN']
        actual_rating = row['Book-Rating']
        predicted_rating = predict_rating(user, isbn, user_item_matrix, similarity_df, k)

        if not np.isnan(predicted_rating):
            predictions.append(predicted_rating)
            actuals.append(actual_rating)

    return np.array(predictions), np.array(actuals)

def mean_absolute_difference(predictions, actuals):
    return np.mean(np.abs(predictions - actuals))

for k in [5, 10, 15, 20, 50, 100]:
    preds, acts = predict_test_ratings(test_data, user_item_matrix_filled, item_similarity_df, k)
    mad = mean_absolute_difference(preds, acts)
    print(f"k = {k}, MAD = {mad:.4f}")

100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 876.19it/s]


k = 5, MAD = 1.2460


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 872.28it/s]


k = 10, MAD = 1.2343


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 864.03it/s]


...

k = 20, MAD = 1.2318


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 863.51it/s]


k = 50, MAD = 1.2328


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 862.37it/s]

k = 100, MAD = 1.2327
